## Train linear probe on embeddings

Load a parquet produced by **`embed.ipynb`** (class token only) or **`embed_cls_patch.ipynb`** (cls + one spatial patch). Set **`data_timestamp`** and **`embedding_parquet`** below (and **`EMBEDDING_STRATEGY`** for logging only). **`feature_columns`** are columns named **`cls{n}`** or **`spatial{n}`**; **`probe_input_dim`** follows from that.


In [ ]:
from datetime import date, datetime
import json
import re
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from sklearn.metrics import average_precision_score, classification_report, f1_score
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import legacy as legacy_optimizers

parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import model_library

%load_ext autoreload
%autoreload 2

# --- Match the embedding notebooks ---
data_timestamp = "2026-03-24T23:15"
data_dir = Path(f"../data/training_patches{data_timestamp}")

# Parquet must match how you embedded: cls_patch (embed_cls_patch.ipynb) vs cls_only (embed.ipynb)
EMBEDDING_STRATEGY = "cls_patch"  # or "cls_only"
embedding_parquet = data_dir / "ssl4eo_cls-patch.parquet"
# embedding_parquet = data_dir / "ssl4eo.parquet"  # cls_only (embed.ipynb)


def _is_embedding_feature_column(name: str) -> bool:
    return bool(re.fullmatch(r"cls\d+", name) or re.fullmatch(r"spatial\d+", name))


def _embedding_feature_column_sort_key(name: str):
    """cls0…cls383 then spatial0…spatial383 (numeric), not lexicographic."""
    m = re.fullmatch(r"(cls|spatial)(\d+)", name)
    if not m:
        return (2, name, 0)
    kind = 0 if m.group(1) == "cls" else 1
    return (kind, int(m.group(2)))


def embedding_feature_columns(df):
    return sorted(
        (c for c in df.columns if _is_embedding_feature_column(c)),
        key=_embedding_feature_column_sort_key,
    )

df = gpd.read_parquet(embedding_parquet)
print(f"Loaded {len(df)} rows from {embedding_parquet} ({EMBEDDING_STRATEGY})")

feature_columns = embedding_feature_columns(df)
probe_input_dim = len(feature_columns)
print(f"probe_input_dim={probe_input_dim} ({len(feature_columns)} columns cls*/spatial*)")


### Inspect embeddings

Data is already loaded above. Re-run the next cell after changing **`embedding_parquet`** / **`data_timestamp`** if needed.


In [ ]:
print(df[["label", "split"]].value_counts())
df.head()


### Probe training

In [ ]:
def make_tf_dataset(df, split: str, *, feature_columns=None, label_column="label",
    split_column="split", batch_size=8, shuffle=True, prefetch=8):
    """Build a tf.data.Dataset for one split of a merged embeddings+meta dataframe."""
    subset = df.loc[df[split_column] == split]
    if subset.empty:
        raise ValueError(f"No rows with {split_column}={split!r}")

    if feature_columns is None:
        feature_columns = embedding_feature_columns(df)

    X = subset[feature_columns].to_numpy(dtype=np.float32)
    y = subset[label_column].to_numpy()

    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        shuffle_buf = max(1, min(len(X), 8192))
        dataset = dataset.shuffle(buffer_size=shuffle_buf, reshuffle_each_iteration=True)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=prefetch)
    return dataset

def flatten_binary_preds(preds):
    """Single-output (N, 1) or (N,) -> shape (N,). Ensemble (N, K) with K > 1 -> mean over K."""
    preds = np.asarray(preds)
    if preds.ndim == 2:
        if preds.shape[1] == 1:
            # sigmoid binary classifier
            preds = preds.squeeze()
        elif preds.shape[1] == 2:
            # softmax binary classifier
            preds = preds[:, 1]
        else:
            # ensemble of M>2 sigmoid models 
            preds = np.mean(preds, axis=1)
            # (Note: M=2 ensemble would be misconstrued as softmax binary)
    return preds

In [ ]:
# Callbacks
earlystop_cb = EarlyStopping(
    monitor="val_loss",
    patience=20,
    #mode="max",
    min_delta=1e-4,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_cb = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=8,
    min_delta=0.005,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
model_id = 'a'
fraction = 0.7            
seed = ord(model_id)
cls_dropout = 0.5

current_date = date.today()
model_name = f"48px_v4.0cls-patch-MLP32{model_id}_drop{cls_dropout}_{current_date.isoformat()}"
model_path = Path("../checkpoints") / f"{model_name}.h5"
assert not model_path.exists(), f"Model {model_path} already exists"

In [ ]:
df_train = df.loc[df["split"] == "train"].copy()
df_train = (
    df_train.sample(frac=fraction, random_state=seed)
    .assign(sample_id=model_id, seed_used=seed)
)
display(df_train.head())
train_ds = make_tf_dataset(df_train, split="train", shuffle=True, feature_columns=feature_columns)
val_ds = make_tf_dataset(df, split="val", shuffle=False, feature_columns=feature_columns)


In [ ]:
model = model_library.MLP_with_targeted_dropout(
    input_dim=probe_input_dim, 
    hidden_layers=(32,), 
    cls_dropout_rate=cls_dropout
)
#model = model_library.MLP(input_dim=probe_input_dim, hidden_layers=(32,))
#model = model_library.LogisticRegression(input_dim=probe_input_dim)
model.compile(
    optimizer=legacy_optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False), 
    metrics=[tf.keras.metrics.BinaryAccuracy(name="acc")],
    run_eagerly=False
)

In [ ]:
# Or reload to continue training
"""
model_name = '48px_v1.1.1SSL4EO-MLP20251021_172507'
model = tf.keras.models.load_model(f'../checkpoints/{model_name}.h5')

model.compile(
    optimizer=legacy_optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False), 
    metrics=[tf.keras.metrics.BinaryAccuracy(name="acc")],
    run_eagerly=True
)
"""

In [ ]:
timestamp = datetime.now().strftime("%H%M%S")
checkpoint_dir = Path("../checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

try: 
    checkpoint_path = checkpoint_dir / f"{model_name}_{timestamp}.h5"
except NameError: 
    checkpoint_path = checkpoint_dir / f"best_model{timestamp}.h5"

# Consider adding something dynamic like this to checkpoint_path: "/model_{epoch:02d}_{val_acc:.4f}.h5" 
checkpoint_cb = ModelCheckpoint(
    filepath=str(checkpoint_path),
    monitor="val_acc",
    save_best_only=True,
    save_weights_only=False,
    mode="max",
    verbose=1
)

In [ ]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=200, 
    batch_size=min(2048, len(df_train)),  
    verbose=1,
    callbacks=[checkpoint_cb, reduce_lr_cb, earlystop_cb]
)

In [ ]:
model.save(model_path)
print(f"Saved {model_path}")

#### Eval

In [ ]:
# Select a split for evaluation, from "val", "test1", etc.
split = "val"
target_names = ['No Mine', 'Mine']
eval_ds = make_tf_dataset(df, split=split, shuffle=False, feature_columns=feature_columns)
y_eval = df.loc[df["split"] == split, "label"].to_numpy()  # Note the labels will only align when shuffle=False


In [ ]:
model_name = '48px_v0.1cls-patch-MLP32a_2026-04-03'; model_path = Path(f'../checkpoints/{model_name}.h5')
model = tf.keras.models.load_model(model_path)
model.summary()

In [ ]:
with tf.device("/CPU:0"):
    preds = model.predict(eval_ds, verbose=1)
preds = flatten_binary_preds(preds)
preds.shape

In [ ]:
def acc_curve(preds, y_true, thresholds=np.arange(.01, 1.01, .01)):
    """Compute accuracy curve as function of threshold"""
    score = [np.sum((preds >= t).astype('int') == y_true) / len(y_true) for t in thresholds]
    plt.plot(thresholds, score)
    plt.ylabel('Success Rate')
    plt.xlabel('Threshold')
    plt.title(f"Optimal Threshold: {thresholds[np.argmax(score)]:.2f} w/ accuracy {score[np.argmax(score)]:.2f}")

acc_curve(preds, y_eval)

In [ ]:
def f1_curve(preds, y_true, thresholds=np.arange(.01, 1.01, .01)):
    """Compute F1 curve."""
    f1s = []
    for t in thresholds:
        y_pred = (preds >= t)
        f1s.append(f1_score(y_true, y_pred))

    fig, ax = plt.subplots()
    ax.plot(thresholds, f1s, label='Patchwise')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('F1 score')
    ax.legend(loc='lower left')
    plt.title(f"Optimal Threshold: {thresholds[np.argmax(f1s)]:.2f} w/ F1 {f1s[np.argmax(f1s)]:.2f}")
    return fig, ax

f1_curve(preds, y_eval)

#### Test evals

In [ ]:
threshold = 0.5

reports = {}
avg_precs = {}
for split in ['val', 'test1', 'test2']:
    eval_ds = make_tf_dataset(df, split=split, shuffle=False, feature_columns=feature_columns)
    y_eval = df.loc[df["split"] == split, "label"].to_numpy()  # Note the labels will only align when shuffle=False
    with tf.device("/CPU:0"):
        preds = model.predict(eval_ds, verbose=1)
    preds = flatten_binary_preds(preds)
    reports.update({
        split: classification_report(y_eval, preds > threshold, target_names=target_names, output_dict=True)
    })
    avg_precs.update({
        split: average_precision_score(y_eval, preds)
    })


In [ ]:
print(f"{model_name} @ t={threshold}")
for split, report in reports.items():
    report = pd.DataFrame(report).transpose()
    print(f"{split}")
    display(f"Avg precision: {avg_precs[split]}")
    display(report.drop([i for i in report.index if 'avg' in i]))

In [ ]:
threshold = 0.5
training_dataset = data_dir.stem

with open(model_path.with_suffix("").as_posix() + f"_config-t{threshold}_testevals.txt", "w") as f:
    f.write(f"Training dataset: {training_dataset}")
    f.write(f"Avg precisions: {avg_precs}")
    f.write(f"\n\nClassification Reports at {threshold}\n")
    f.write(json.dumps(reports, indent=4))

#### View model errors

Select a single evaluation set and run the eval code above to establish y_eval and preds.

In [ ]:
split = 'val'
threshold = 0.8

df_split = df[df.split == split]
df_split['pred'] = preds
misses = df_split[df_split.label != (preds > threshold).astype('int')]
hits = df_split[df_split.label == (preds > threshold).astype('int')]
#riverbanks = df_split.cx[-73:-66, -10.5:-6]
#riverbanks = df_split.cx[-68:-66, -16:-13]
#riverbanks = df_split.cx[-66:-63, -18:-16]
print(f"{len(misses)} incorrect predictions at t={threshold}")
misses.head()

In [ ]:
rgbs = []

to_vis = misses.copy()
#to_vis = hits.sample(n=100, random_state=43)
#to_vis = riverbanks.copy()

def get_rgb(pixels: np.ndarray) -> np.ndarray:
    """Convert Sentinel-2 reflectance -> uint8 RGB w/ constrast stretch."""
    rgb = pixels[..., [3, 2, 1]].astype(np.float32)
    rgb = np.clip(rgb / 3000.0, 0, 1)
    return (rgb * 255).astype(np.uint8)

num_img = int(np.ceil(np.sqrt(len(to_vis))))
fig, axes = plt.subplots(num_img, num_img, figsize=(num_img, num_img), dpi=300)

fig.subplots_adjust(wspace=0.6, hspace=0.6)
axes = axes.flatten()
for ax in axes:
    ax.axis('off')

for ax, (i, row) in zip(axes, to_vis.iterrows()):
    with rasterio.open(row.path) as f:
        img = f.read()
        img = np.moveaxis(img, 0, -1)
        rgb = get_rgb(img)
        ax.imshow(rgb)
        ax.set_title(f"{i}: label: {row.label}\npred: {row.pred:0.3f}", fontsize='x-small')

In [ ]:
with rasterio.open(to_vis.loc[17232].path) as f:
    img = f.read()
    img = np.moveaxis(img, 0, -1)
    rgb = get_rgb(img)

plt.imshow(rgb)

In [ ]:
to_vis.loc[16792]